# Resume Perser

#### Dependencies

In [ ]:
pip install pypdf

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.0/12.8 MB 330.3 kB/s eta 0:00:39
     --------------------------------------- 0.1/12.8 MB 871.5 kB/s eta 0:00:15
      --------------------------------------- 0.3/12.8 MB 2.0 MB/s eta 0:00:07
     --- ------------------------------------ 1.0/12.8 MB 5.1 MB/s eta 0:00:03
     ------- -------------------------------- 2.3/12.8 MB 9.9 MB/s eta 0:00:02
     ------------------- -------------------- 6.1/12.8 MB 21.6 MB/s eta 0:00:01
     ---------------------- ----------------- 7.1/12.8 MB 21.7 MB/s eta 0:00:01
     ------------------------ --------------- 7.9/12.8 MB 22.0 MB/s eta 0:00:01
     ---------------------------- ----------- 9.3/12.8 MB 22.0 MB/s eta 0:00:01
     ------------------------------------- - 12.4/12.8 MB 43.7 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 38.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now 

ERROR: Pipe to stdout was broken


#### Global Variables, Imports and Functions

In [ ]:
from pypdf import PdfReader
import os
import re
import json

output_file = "extracted_data.json"
pdf_directory = "dataset/"

Keywords = [
    "education",
    "summary",
    "accomplishments",
    "executive profile",
    "professional profile",
    "personal profile",
    "work background",
    "academic profile",
    "other activities",
    "qualifications",
    "experience",
    "interests",
    "skills",
    "achievements",
    "publications",
    "publication",
    "certifications",
    "workshops",
    "projects",
    "internships",
    "trainings",
    "hobbies",
    "overview",
    "objective",
    "position of responsibility",
    "jobs"
]

In [ ]:
def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

In [ ]:
def get_email_addresses(string):
    r = re.compile(r'[\w\.-]+@[\w\.-]+')
    return r.findall(string)

In [ ]:
def get_phone_numbers(string):
    r = re.compile(r'(\+?\d{1,4}[\s.-]?)?(\(?\d{1,4}?\)?[\s.-]?)?(\d{3,4}[\s.-]?\d{3,9})')
    phone_numbers = r.finditer(string)
    return [num.group() for num in phone_numbers]

In [ ]:
def extract_name(text):
    header_text = " ".join(text.strip().split('\n')[:2])
    name_pattern = r'(([A-Z][a-z]+|[A-Z]{2,})(?:\s([A-Z][a-z]+|[A-Z]{2,})){1,2})'    
    matches = re.findall(name_pattern, header_text)

    blacklist = ["Resume", "Curriculum", "Vitae", "Page", "Address", "Contact"]
    
    for match in matches:
        if not any(word in match for word in blacklist):
            return match
            
    return None

In [ ]:
def extract_header_sections(text):
    content = {}
    indices = []
    keys = []
    for key in Keywords:
        try:
            content[key] = text[text.index(key) + len(key):]
            indices.append(text.index(key))
            keys.append(key)
        except:
            pass
    zipped_lists = zip(indices, keys)
    sorted_pairs = sorted(zipped_lists)
    sorted_pairs

    tuples = zip(*sorted_pairs)
    indices, keys = [list(tuple) for tuple in tuples]

    content = []
    for idx in range(len(indices)):
        if idx != len(indices)-1:
            content.append(text[indices[idx]: indices[idx+1]])
        else:
            content.append(text[indices[idx]: ])
    
    return indices, keys, content

#### Main Body

In [41]:
# We sould get all pdf files from a directory
pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith('.pdf')]

print(f"Found {len(pdf_files)} PDF files.")

pdf_mapper = {}
for pdf_file in pdf_files:
    file_path = os.path.join(pdf_directory, pdf_file)

    text = extract_text_from_pdf(file_path)
    print(f"Extracted text from {pdf_file}:")
    
    pdf_mapper[pdf_file] = {}

    emails = get_email_addresses(text)
    pdf_mapper[pdf_file]['emails'] = emails
    print(f"  Found emails: {emails}")

    phones = get_phone_numbers(text)
    pdf_mapper[pdf_file]['phone_numbers'] = phones
    print(f"  Found phone numbers: {phones}")

    name = extract_name(text)
    pdf_mapper[pdf_file]['name'] = name
    print(f"  Extracted name: {name}")

    text = text.replace("\n"," ")
    text = text.replace("[^a-zA-Z0-9]", " ");  
    re.sub('\W+','', text)
    section_indices, section_keys, section_contents = extract_header_sections(text.lower())
    for i in range(len(section_indices)):
        print(f"  Found section {section_keys[i]}: {section_contents[i]}")
        pdf_mapper[pdf_file][section_keys[i]] = section_contents[i]



<>:29: SyntaxWarning: invalid escape sequence '\W'
<>:29: SyntaxWarning: invalid escape sequence '\W'
C:\Users\BaddD\AppData\Local\Temp\ipykernel_3964\2487017573.py:29: SyntaxWarning: invalid escape sequence '\W'
  re.sub('\W+','', text)


Found 7 PDF files.
Extracted text from 1901841_RESUME.pdf:
  Found emails: ['anuvagoyal111@gmail.com']
  Found phone numbers: ['+91 9520349542']
  Extracted name: ('ANUVA GOYAL', 'ANUVA', 'GOYAL')
  Found section objective: objective    energetic, innovative engineering undergraduate, passionate about machine learning, nlp and deep learning for  solving real-world problems, aiming to work in an organization providing great learning 
  Found section experience: experience and growth  opportunities for mutual benefit.    
  Found section education: education   qualification institute cgpa year of completion     b.tech. dayalbagh educational institute, cgpa 9.35 2023       (electrical engineering (till 4        dayalbagh, agra pursuing      specialization in computer science) semesters)            xii st. clare’s senior secondary school, agra  94% 2019      x st. clare’s senior secondary school, agra  cgpa 10 2017      
  Found section internships: internships and 
  Found section trainin

In [ ]:
with open(output_file, "w") as outfile:
    json.dump(pdf_mapper, outfile)

a_file = open(output_file, "r")
a_json = json.load(a_file)
pretty_json = json.dumps(a_json, indent=4)
a_file.close()
print(pretty_json)

{
    "1901841_RESUME.pdf": {
        "emails": [
            "anuvagoyal111@gmail.com"
        ],
        "phone_numbers": [
            "+91 9520349542"
        ],
        "name": [
            "ANUVA GOYAL",
            "ANUVA",
            "GOYAL"
        ],
        "objective": "objective    energetic, innovative engineering undergraduate, passionate about machine learning, nlp and deep learning for  solving real-world problems, aiming to work in an organization providing great learning ",
        "experience": "experience and growth  opportunities for mutual benefit.    ",
        "education": "education   qualification institute cgpa year of completion     b.tech. dayalbagh educational institute, cgpa 9.35 2023       (electrical engineering (till 4        dayalbagh, agra pursuing      specialization in computer science) semesters)            xii st. clare\u2019s senior secondary school, agra  94% 2019      x st. clare\u2019s senior secondary school, agra  cgpa 10 2017      ",
  